# GLiNER2 Synthetic Data Generator Demo

This notebook demonstrates the `DataGenerator` class from `generate.py`, which turns a natural language task description into synthetic GLiNER2-style training examples.

By default, the generator infers tasks using lightweight rules, but you can also enable an optional LLM-based inference mode backed by a local Ollama `llama3.2` model.

It also includes a scaffold for fine-tuning `fastino/gliner2-base-v1` on the generated data using the GLiNER2 training API.

In [1]:
from pprint import pprint

from generate import DataGenerator

generator = DataGenerator(seed=42)

## Single-task examples

We first generate examples for each of the four GLiNER2 task types:

- NER
- Classification
- Relation extraction
- JSON extraction


In [2]:
single_task_descriptions = [
    "Extract company, person and location names from short tech news snippets.",  # NER
    "Classify customer review sentiment into positive, negative and neutral.",    # Classification
    "Identify works_for and lives_in relations between people and organizations.", # Relation extraction
    "Extract product name, storage and price into a JSON structure.",             # JSON extraction
]

for desc in single_task_descriptions:
    print("==== Description ====")
    print(desc)
    examples = generator.generate(desc, n=3)
    print("First example:")
    pprint(examples[0])
    print()

==== Description ====
Extract company, person and location names from short tech news snippets.
First example:
{'input': 'Acme Corp, based in Toronto, recently hired Bob Smith to lead a new '
          'project.',
 'output': {'entities': {'company': ['Acme Corp'],
                         'location': ['Toronto'],
                         'person': ['Bob Smith']}}}

==== Description ====
Classify customer review sentiment into positive, negative and neutral.
First example:
{'input': 'The mobile app worked flawlessly and exceeded my expectations.',
 'output': {'classifications': [{'labels': ['positive', 'negative', 'neutral'],
                                 'task': 'sentiment',
                                 'true_label': ['positive']}]}}

==== Description ====
Identify works_for and lives_in relations between people and organizations.
First example:
{'input': 'Sara has been living in New York ever since joining Aurora '
          'Robotics.',
 'output': {'relations': [{'works_for': 

## Multi-task examples

Now we use composite task descriptions that implicitly require multiple task types to be active at once.


In [3]:
multi_task_descriptions = [
    # NER + classification
    "From each customer review, extract company names and classify overall sentiment into positive, negative or neutral.",
    # JSON extraction + classification
    "Extract a JSON product record (name, storage, price) from each sentence and classify whether the tone is positive, negative or neutral.",
]

for desc in multi_task_descriptions:
    print("==== Multi-task description ====")
    print(desc)
    examples = generator.generate(desc, n=4)
    print("Example 0:")
    pprint(examples[0])
    print("Example 1:")
    pprint(examples[1])
    print()

==== Multi-task description ====
From each customer review, extract company names and classify overall sentiment into positive, negative or neutral.
Example 0:
{'input': 'Fastino AI reported record quarterly revenue this year. Customer '
          'support offers great value for the subscription price.',
 'output': {'classifications': [{'labels': ['positive', 'negative', 'neutral'],
                                 'task': 'sentiment',
                                 'true_label': ['positive']}],
            'entities': {'company': ['Fastino AI']}}}
Example 1:
{'input': 'Shares of NeuralForge rose sharply following the product launch. '
          'The pricing model kept crashing in the middle of important work.',
 'output': {'classifications': [{'labels': ['positive', 'negative', 'neutral'],
                                 'task': 'sentiment',
                                 'true_label': ['negative']}],
            'entities': {'company': ['NeuralForge']}}}

==== Multi-task descrip

## Optional: LLM-backed task inference (Ollama `llama3.2`)

If you have Ollama running locally with a `llama3.2` model available, you can enable **LLM-based task inference**.

This section:
- checks whether the local Ollama server is reachable
- compares rule-based vs LLM-inferred task settings
- runs `generate()` with `task_inference_mode="llm"` to confirm end-to-end compatibility

In [4]:
import os
from dataclasses import asdict

import requests

from generate import DataGenerator

# Set the Ollama host (adjust to match your environment; defaults to localhost:11434).
os.environ["OLLAMA_HOST"] = "172.21.112.1:11434"
host = os.environ["OLLAMA_HOST"]
base_url = f"http://{host}" if not host.startswith(("http://", "https://")) else host.rstrip("/")
TIMEOUT = (3, 60)  # (connect_s, read_s)

# Verify Ollama is reachable and list available models.
try:
    resp = requests.post(
        f"{base_url}/api/generate",
        json={"model": "llama3.2", "prompt": "Hello from WSL!", "stream": False},
        timeout=TIMEOUT,
    )
    resp.raise_for_status()
    print("Ollama reachable. Response:", resp.json().get("response", "")[:80])
    models = [m["name"] for m in requests.get(f"{base_url}/api/tags", timeout=TIMEOUT[0]).json().get("models", [])]
    print("Available models:", models)
except requests.exceptions.Timeout:
    print("Ollama request timed out")
except requests.exceptions.RequestException as e:
    print("Ollama unavailable:", e)

# Compare rule-based vs LLM task-type inference on the same descriptions.
rules_gen = DataGenerator(task_inference_mode="rules")
llm_gen = DataGenerator(task_inference_mode="llm", llm_model="llama3.2")

descriptions_to_test = [
    "Extract company, person and location names from short tech news snippets.",
    "Classify customer review sentiment into positive, negative and neutral.",
    "Identify works_for and lives_in relations between people and organizations.",
    "Extract product name, storage and price into a JSON structure.",
    "Extract company names and classify sentiment.",
    "Extract a JSON product record (name, storage, price) and classify the tone as positive, negative, or neutral.",
]

for desc in descriptions_to_test:
    print(f"\n==== {desc}")
    print("rules:", asdict(rules_gen._infer_tasks_rules(desc)))
    llm_result = llm_gen._infer_tasks_via_llm(desc)
    print("llm:  ", asdict(llm_result) if llm_result is not None else "<unavailable>")

Ollama reachable. Response: Hello from the other side... of the terminal, that is! Welcome to our conversati
Available models: ['llama3.2:latest', 'huihui_ai/llama3.2-abliterate:3b']

==== Extract company, person and location names from short tech news snippets.
rules: {'ner': True, 'classification': False, 'relation_extraction': False, 'json_extraction': False, 'classification_task_name': None, 'classification_labels': None, 'ner_entity_labels': None}
llm:   {'ner': True, 'classification': False, 'relation_extraction': False, 'json_extraction': False, 'classification_task_name': None, 'classification_labels': None, 'ner_entity_labels': None}

==== Classify customer review sentiment into positive, negative and neutral.
rules: {'ner': False, 'classification': True, 'relation_extraction': False, 'json_extraction': False, 'classification_task_name': 'sentiment', 'classification_labels': ['positive', 'negative', 'neutral'], 'ner_entity_labels': None}
llm:   <unavailable>

==== Identify wor

In [5]:
from generate import DataGenerator

# End-to-end generation using LLM inference mode.
# Falls back to rule-based inference if Ollama is unavailable.
llm_generator = DataGenerator(seed=7, task_inference_mode="llm", llm_model="llama3.2")

desc = "From each customer review, extract company names and classify overall sentiment into positive, negative or neutral."
examples = llm_generator.generate(desc, n=3)

print("\n==== LLM-mode generate() output (example 0) ====")
pprint(examples[0])


==== LLM-mode generate() output (example 0) ====
{'input': 'Innotech reported record quarterly revenue this year. Customer '
          'support was incredibly smooth from start to finish.',
 'output': {'classifications': [{'labels': ['positive', 'negative', 'neutral'],
                                 'task': 'sentiment',
                                 'true_label': ['positive']}],
            'entities': {'company': ['Innotech']}}}


## Bonus: Fine-tune GLiNER2 on generated data (scaffold)

The following cells show a minimal workflow for fine-tuning `fastino/gliner2-base-v1` on a synthetic sentiment classification task using the generated JSONL data.

This is **scaffold code**: you may need to adjust batch sizes, epochs, and device configuration depending on your hardware.


In [6]:
import json
from pathlib import Path

from generate import DataGenerator

data_dir = Path("synthetic_data")
data_dir.mkdir(exist_ok=True)

sentiment_description = (
    "Classify customer review sentiment into positive, negative and neutral. "
    "Make the examples diverse across topics."
)

SENTIMENT_LABELS = ["positive", "negative", "neutral"]

# Adjust these to generate a larger dataset.
# The sentiment template pool supports up to 8 subjects × 8 outcomes = 64 unique
# examples per label, so the practical ceiling is 3 × 64 = 192 total unique examples.
TRAIN_N = 80
EVAL_N = 20


def generate_balanced_unique(
    generator: DataGenerator,
    description: str,
    n: int,
    labels: list[str],
    exclude_keys: set[str] | None = None,
    max_attempts: int = 0,
) -> tuple[list[dict], set[str]]:
    """Generate exactly n structurally unique examples with balanced labels.

    Each label receives floor(n/len(labels)) or floor(n/len(labels))+1 examples,
    matching the cycling distribution used internally by DataGenerator. The
    exclude_keys set prevents any example fingerprint from appearing in both
    the training and evaluation splits.

    Raises ValueError if max_attempts iterations are exhausted before the
    quotas are filled (indicates the template pool is too small for the
    requested n — lower n or switch to example_generation_mode='llm').
    """
    if max_attempts <= 0:
        max_attempts = max(n * 10, 100)

    seen: set[str] = set(exclude_keys or [])
    base = n // len(labels)
    quota = {l: base + (1 if i < n % len(labels) else 0) for i, l in enumerate(labels)}
    bucket: dict[str, list[dict]] = {l: [] for l in labels}
    attempts = 0

    while any(len(bucket[l]) < quota[l] for l in labels):
        attempts += 1
        if attempts > max_attempts:
            filled = sum(len(v) for v in bucket.values())
            raise ValueError(
                f"generate_balanced_unique: could not collect {n} unique examples after "
                f"{max_attempts} attempts ({filled}/{n} collected). "
                f"The template pool may be exhausted for this task. "
                f"Lower n or switch to example_generation_mode='llm'."
            )
        still_needed = sum(max(0, quota[l] - len(bucket[l])) for l in labels)
        batch = generator.generate(description, n=still_needed * 3)
        for ex in batch:
            key = json.dumps(ex, sort_keys=True, ensure_ascii=False)
            if key in seen:
                continue
            clss = ex.get("output", {}).get("classifications") or []
            if not clss:
                continue
            lbl = (clss[0].get("true_label") or [None])[0]
            if lbl not in bucket or len(bucket[lbl]) >= quota[lbl]:
                continue
            seen.add(key)
            bucket[lbl].append(ex)

    # Round-robin interleave so the sequence preserves label cycling order.
    result: list[dict] = []
    for i in range(base):
        for l in labels:
            result.append(bucket[l][i])
    for l in labels[: n % len(labels)]:
        result.append(bucket[l][base])

    return result, seen


# Training set: seed=123.
train_gen = DataGenerator(seed=123)
train_examples, train_keys = generate_balanced_unique(
    train_gen, sentiment_description, n=TRAIN_N, labels=SENTIMENT_LABELS
)

# Eval set: seed=456, independently generated. exclude_keys=train_keys
# ensures no example can appear in both splits.
eval_gen = DataGenerator(seed=456)
eval_examples, _ = generate_balanced_unique(
    eval_gen, sentiment_description, n=EVAL_N, labels=SENTIMENT_LABELS,
    exclude_keys=train_keys,
)

train_path = data_dir / "train.jsonl"
eval_path = data_dir / "eval.jsonl"

with train_path.open("w", encoding="utf-8") as f:
    for ex in train_examples:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

with eval_path.open("w", encoding="utf-8") as f:
    for ex in eval_examples:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

train_path, eval_path

(PosixPath('synthetic_data/train.jsonl'),
 PosixPath('synthetic_data/eval.jsonl'))

In [7]:
from gliner2 import GLiNER2  # pyright: ignore[reportMissingImports]
from gliner2.training.trainer import GLiNER2Trainer, TrainingConfig  # pyright: ignore[reportMissingImports]

model_name = "fastino/gliner2-base-v1"
model = GLiNER2.from_pretrained(model_name)

config = TrainingConfig(
    output_dir="./output_gliner2_sentiment",
    experiment_name="gliner2_sentiment_demo",
    num_epochs=3,
    batch_size=8,
    encoder_lr=5e-5,
    task_lr=5e-5,
)

trainer = GLiNER2Trainer(model, config)
trainer.train(train_data=str(train_path), eval_data=str(eval_path))

You are using a model of type extractor to instantiate a model of type . This is not supported for all configurations of models and can yield errors.


🧠 Model Configuration
Encoder model      : microsoft/deberta-v3-base
Counting layer     : count_lstm_v2
Token pooling      : first


Mixed precision disabled on CPU
2026-03-03 22:31:45 - INFO - gliner2.training.trainer - LoRA is disabled
Validating records: 100%|██████████| 80/80 [00:00<00:00, 155705.02record/s]
/home/adamp/Programs/GLiNER2_synthetic_data/.venv/lib/python3.12/site-packages/gliner2/training/trainer.py:900: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = GradScaler(enabled=self.config.fp16)
2026-03-03 22:31:45 - INFO - gliner2.training.trainer - ***** Running Training *****
2026-03-03 22:31:45 - INFO - gliner2.training.trainer -   Num examples = 80
2026-03-03 22:31:45 - INFO - gliner2.training.trainer -   Num epochs = 3
2026-03-03 22:31:45 - INFO - gliner2.training.trainer -   Batch size = 8
2026-03-03 22:31:45 - INFO - gliner2.training.trainer -   Gradient accumulation steps = 1
2026-03-03 22:31:45 - INFO - gliner2.training.trainer -   Effective batch size = 8
2026-03-03 22:31:45 - INFO - gliner2.training.t

Training:   0%|          | 0/30 [00:00<?, ?it/s]

/home/adamp/Programs/GLiNER2_synthetic_data/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/home/adamp/Programs/GLiNER2_synthetic_data/.venv/lib/python3.12/site-packages/gliner2/training/trainer.py:947: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp, dtype=amp_dtype):
2026-03-03 22:32:04 - INFO - gliner2.training.trainer - Epoch 1/3 - Loss: 8.2234
2026-03-03 22:32:25 - INFO - gliner2.training.trainer - Epoch 2/3 - Loss: 1.6257
2026-03-03 22:32:44 - INFO - gliner2.training.trainer - Epoch 3/3 - Loss: 0.0086
2026-03-03 22:32:47 - INFO - gliner2.training.trainer - 💾 Saved full checkpoint 'final' | step 30 | epoch 3.0 | 208,476,821 params | 803.6MB | 2.9s


{'total_steps': 30,
 'total_epochs': 3,
 'total_time_seconds': 62.032285928726196,
 'samples_per_second': 3.8689530202990583,
 'best_metric': inf,
 'train_metrics_history': [{'loss': 11.44719123840332,
   'classification_loss': 11.44719123840332,
   'structure_loss': 0.0,
   'count_loss': 0.0,
   'learning_rate': 1.6666666666666667e-05,
   'epoch': 0.0,
   'step': 1,
   'samples_seen': 8,
   'throughput': 2.530504730928623},
  {'loss': 28.043062210083008,
   'classification_loss': 28.043062210083008,
   'structure_loss': 0.0,
   'count_loss': 0.0,
   'learning_rate': 3.3333333333333335e-05,
   'epoch': 0.1,
   'step': 2,
   'samples_seen': 16,
   'throughput': 2.995168195533031},
  {'loss': 12.519535064697266,
   'classification_loss': 12.519535064697266,
   'structure_loss': 0.0,
   'count_loss': 0.0,
   'learning_rate': 5e-05,
   'epoch': 0.2,
   'step': 3,
   'samples_seen': 24,
   'throughput': 3.2462776981929378},
  {'loss': 4.679790019989014,
   'classification_loss': 4.679790019

### Evaluation metrics

We evaluate on the held-out synthetic set with two kinds of metrics:

- **Classification accuracy**: fraction of examples where the predicted label exactly matches the gold label. We report overall accuracy for both the base and fine-tuned model, and optionally per-label accuracy (useful when labels are imbalanced).
- **Span match (entity-level)** when the eval data contains NER: we treat each `(label, span_text)` pair as one span. A predicted span matches gold if the same text is listed for that label in the gold output. We aggregate **precision** (matched predictions / total predictions), **recall** (matched gold / total gold), and **F1** (harmonic mean of P and R). These are micro-averaged over all entity types.

In [8]:
from collections import defaultdict


def _gold_entities_set(output_entities: dict) -> set[tuple[str, str]]:
    """Flatten output['entities'] to a set of (label, span_text) pairs."""
    return {
        (label, s.strip())
        for label, spans in (output_entities or {}).items()
        for s in spans
        if isinstance(s, str) and s.strip()
    }


def _pred_entities_set(pred: dict) -> set[tuple[str, str]]:
    """Flatten a GLiNER2 entity prediction dict to a set of (label, span_text) pairs."""
    entities = pred.get("entities") if isinstance(pred, dict) else {}
    return {
        (label, s.strip())
        for label, spans in (entities or {}).items()
        for s in (spans if isinstance(spans, list) else [])
        if isinstance(s, str) and s.strip()
    }


def span_metrics(gold_set: set, pred_set: set) -> dict:
    """Micro-averaged P/R/F1 for exact (label, span_text) match."""
    if not gold_set and not pred_set:
        return {"precision": 1.0, "recall": 1.0, "f1": 1.0}
    matched = len(gold_set & pred_set)
    prec = matched / len(pred_set) if pred_set else 0.0
    rec = matched / len(gold_set) if gold_set else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    return {"precision": prec, "recall": rec, "f1": f1}


# Load eval examples from the independently generated held-out set.
eval_examples = []
with eval_path.open("r", encoding="utf-8") as f:
    for line in f:
        eval_examples.append(json.loads(line))

has_classification = any(ex.get("output", {}).get("classifications") for ex in eval_examples)
has_entities = any(ex.get("output", {}).get("entities") for ex in eval_examples)

# Read task config from the first classification example.
cls_task_name = cls_labels = None
for ex in eval_examples:
    clss = ex.get("output", {}).get("classifications") or []
    if clss:
        cls_task_name = clss[0].get("task", "classification")
        cls_labels = clss[0].get("labels", ["positive", "negative", "neutral"])
        break

base_cls_correct = ft_cls_correct = 0
per_label_gold: dict[str, int] = defaultdict(int)
per_label_base_correct: dict[str, int] = defaultdict(int)
per_label_ft_correct: dict[str, int] = defaultdict(int)

gold_span_sets: list[set] = []
base_pred_span_sets: list[set] = []
ft_pred_span_sets: list[set] = []

# Load the base model separately so both base and fine-tuned can be compared.
base_model = GLiNER2.from_pretrained(model_name)  # pyright: ignore[reportMissingImports]

for ex in eval_examples:
    text = ex["input"]
    out = ex.get("output", {})

    if has_classification and out.get("classifications"):
        true_cls = out["classifications"][0]["true_label"][0]
        per_label_gold[true_cls] += 1
        tasks = {cls_task_name: {"labels": cls_labels}}

        base_cls = base_model.classify_text(text, tasks=tasks, threshold=0.5, format_results=True, include_confidence=False).get(cls_task_name)
        ft_cls = model.classify_text(text, tasks=tasks, threshold=0.5, format_results=True, include_confidence=False).get(cls_task_name)

        if base_cls == true_cls:
            base_cls_correct += 1
            per_label_base_correct[true_cls] += 1
        if ft_cls == true_cls:
            ft_cls_correct += 1
            per_label_ft_correct[true_cls] += 1

    if has_entities and out.get("entities"):
        gold_span_sets.append(_gold_entities_set(out["entities"]))
        entity_labels = list(out["entities"].keys())
        base_pred_span_sets.append(_pred_entities_set(base_model.extract_entities(text, entity_labels)))
        ft_pred_span_sets.append(_pred_entities_set(model.extract_entities(text, entity_labels)))

n_cls = sum(per_label_gold.values()) if has_classification else 0
results = {}

if has_classification and n_cls > 0:
    results["classification"] = {
        "num_examples": n_cls,
        "base_accuracy": base_cls_correct / n_cls,
        "fine_tuned_accuracy": ft_cls_correct / n_cls,
        "per_label": {
            label: {
                "count": g,
                "base_accuracy": per_label_base_correct[label] / max(1, g),
                "fine_tuned_accuracy": per_label_ft_correct[label] / max(1, g),
            }
            for label, g in per_label_gold.items()
        },
    }

if has_entities and gold_span_sets:
    all_gold = set().union(*gold_span_sets)
    all_base = set().union(*base_pred_span_sets)
    all_ft = set().union(*ft_pred_span_sets)
    results["span_match"] = {
        "num_spans_gold": len(all_gold),
        "base": span_metrics(all_gold, all_base),
        "fine_tuned": span_metrics(all_gold, all_ft),
    }

results

2026-03-03 22:32:47 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/fastino/gliner2-base-v1/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-03 22:32:47 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/fastino/gliner2-base-v1/283f4af5e598631a5352b8c388b6906853146f07/config.json "HTTP/1.1 200 OK"
2026-03-03 22:32:47 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/fastino/gliner2-base-v1/resolve/main/encoder_config/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-03 22:32:47 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/fastino/gliner2-base-v1/283f4af5e598631a5352b8c388b6906853146f07/encoder_config%2Fconfig.json "HTTP/1.1 200 OK"
2026-03-03 22:32:47 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/fastino/gliner2-base-v1/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-03 22:32:48 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/api/r

🧠 Model Configuration
Encoder model      : microsoft/deberta-v3-base
Counting layer     : count_lstm_v2
Token pooling      : first


{'classification': {'num_examples': 20,
  'base_accuracy': 0.9,
  'fine_tuned_accuracy': 1.0,
  'per_label': {'positive': {'count': 7,
    'base_accuracy': 1.0,
    'fine_tuned_accuracy': 1.0},
   'negative': {'count': 7, 'base_accuracy': 1.0, 'fine_tuned_accuracy': 1.0},
   'neutral': {'count': 6,
    'base_accuracy': 0.6666666666666666,
    'fine_tuned_accuracy': 1.0}}}}

In [9]:
# Summarise and compare metrics
if "classification" in results:
    c = results["classification"]
    print("Classification (accuracy)")
    print(f"  Base:       {c['base_accuracy']:.2%}  Fine-tuned: {c['fine_tuned_accuracy']:.2%}")
    if "per_label" in c:
        for label, stats in c["per_label"].items():
            print(f"  [{label}] n={stats['count']}  base={stats['base_accuracy']:.2%}  ft={stats['fine_tuned_accuracy']:.2%}")
if "span_match" in results:
    s = results["span_match"]
    print("Span match (entity-level micro P/R/F1)")
    print(f"  Base:       P={s['base']['precision']:.3f} R={s['base']['recall']:.3f} F1={s['base']['f1']:.3f}")
    print(f"  Fine-tuned: P={s['fine_tuned']['precision']:.3f} R={s['fine_tuned']['recall']:.3f} F1={s['fine_tuned']['f1']:.3f}")

Classification (accuracy)
  Base:       90.00%  Fine-tuned: 100.00%
  [positive] n=7  base=100.00%  ft=100.00%
  [negative] n=7  base=100.00%  ft=100.00%
  [neutral] n=6  base=66.67%  ft=100.00%
